# Stage 1. Data Preparation (Formula 1)

In this notebook, I prepared the data for the future model.

What I did:
1. Loaded the main tables from the Kaggle F1 Dataset.
2. Cleaned the data and converted the features into a convenient format.
3. Explored the main distributions and relationships between features (EDA).
4. Built the final dataset `f1_final_dataset.csv` for the next stages.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.style.use('ggplot')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully")

In [ ]:
files = {
    'circuits': 'circuits.csv',
    'constructor_results': 'constructor_results.csv',
    'constructor_standings': 'constructor_standings.csv',
    'constructors': 'constructors.csv',
    'driver_standings': 'driver_standings.csv',
    'drivers': 'drivers.csv',
    'lap_times': 'lap_times.csv',
    'pit_stops': 'pit_stops.csv',
    'qualifying': 'qualifying.csv',
    'races': 'races.csv',
    'results': 'results.csv',
    'seasons': 'seasons.csv',
    'sprint_results': 'sprint_results.csv',
    'status': 'status.csv'
}

In [ ]:
# Data loading
data = {}
for name, path in files.items():
    try:
        data[name] = pd.read_csv(path)
        print(f"✓ {name}: {data[name].shape[0]} rows, {data[name].shape[1]} columns")
    except FileNotFoundError:
        print(f"✗ File {path} not found")
    except Exception as e:
        print(f"✗ Error while loading {name}: {e}")

In [ ]:
# Race results analysis
results_df = data['results']
print("=== RESULTS.CSV ===")
print(f"Total records: {len(results_df)}")
print(f"Unique races: {results_df['raceId'].nunique()}")
print(f"Unique drivers: {results_df['driverId'].nunique()}")
print(f"Unique constructors: {results_df['constructorId'].nunique()}")

In [ ]:
# Finishing position analysis
print("\n=== Position analysis ===")
print(f"position data type: {results_df['position'].dtype}")
print(f"Unique values in position: {results_df['position'].unique()[:20]}")

In [ ]:
# Converting position to numeric format
results_df['position_num'] = pd.to_numeric(results_df['position'], errors='coerce')
print(f"\nAfter numeric conversion:")
print(f"Missing values (DNF, DNS, etc.): {results_df['position_num'].isna().sum()}")
print(f"Finished races: {results_df['position_num'].notna().sum()}")

In [ ]:
# Visualization of finishing position distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

In [ ]:
# Histogram of finishing positions
axes[0].hist(results_df['position_num'].dropna(), bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Finishing position')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of finishing positions')
axes[0].axvline(results_df['position_num'].mean(), color='red', linestyle='--', label=f'Mean: {results_df["position_num"].mean():.1f}')
axes[0].legend()

In [ ]:
# Box plot of positions
results_df.boxplot(column='position_num', ax=axes[1])
axes[1].set_ylabel('Position')
axes[1].set_title('Box plot of finishing positions')

In [ ]:
# Status analysis (retirements, etc.)
status_counts = results_df['statusId'].value_counts().head(10)
axes[2].bar(range(len(status_counts)), status_counts.values)
axes[2].set_xticks(range(len(status_counts)))
axes[2].set_xticklabels(status_counts.index, rotation=45)
axes[2].set_xlabel('Status ID')
axes[2].set_ylabel('Count')
axes[2].set_title('Top 10 finish statuses')
plt.tight_layout()
plt.show()

In [ ]:
# Status analysis with names
status_df = data['status']
results_with_status = results_df.merge(status_df, on='statusId')
status_summary = results_with_status['status'].value_counts().head(15)

plt.figure(figsize=(12, 6))
status_summary.plot(kind='bar')
plt.title('Top 15 finish statuses')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("Main retirement reasons:")
for status, count in status_summary.head(10).items():
    print(f"  {status}: {count} ({count/len(results_with_status)*100:.1f}%)")

In [ ]:
# Qualifying analysis
quali_df = data['qualifying']
print("=== QUALIFYING.CSV ===")
print(f"Total records: {len(quali_df)}")
print(f"Missing values in q1: {quali_df['q1'].isna().sum()}")
print(f"Missing values in q2: {quali_df['q2'].isna().sum()}")
print(f"Missing values in q3: {quali_df['q3'].isna().sum()}")

In [ ]:
# Converting position to numbers
quali_df['quali_position'] = pd.to_numeric(quali_df['position'], errors='coerce')

In [ ]:
# Merging results with qualifying data
results_quali = results_df.merge(
    quali_df[['raceId', 'driverId', 'quali_position']],
    on=['raceId', 'driverId'],
    how='left'
)

In [ ]:
# Removing missing values
results_quali_clean = results_quali.dropna(subset=['position_num', 'quali_position'])

In [ ]:
# Relationship visualization
plt.figure(figsize=(10, 6))
plt.scatter(results_quali_clean['quali_position'],
           results_quali_clean['position_num'],
           alpha=0.3, s=10)
plt.xlabel('Qualifying position')
plt.ylabel('Finishing position')
plt.title('Relationship between qualifying and finish')

In [ ]:
# Regression line
z = np.polyfit(results_quali_clean['quali_position'],
               results_quali_clean['position_num'], 1)
p = np.poly1d(z)
plt.plot(results_quali_clean['quali_position'].sort_values(),
         p(results_quali_clean['quali_position'].sort_values()),
         "r--", alpha=0.8, label=f'Trend: y = {z[0]:.2f}x + {z[1]:.2f}')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation
correlation = results_quali_clean['quali_position'].corr(results_quali_clean['position_num'])
print(f"Correlation between qualifying and finish: {correlation:.3f}")

In [ ]:
# Driver analysis
drivers_df = data['drivers']
print("=== DRIVERS.CSV ===")
print(f"Total drivers: {len(drivers_df)}")
print(f"Nationalities: {drivers_df['nationality'].nunique()}")

In [ ]:
# Birth date conversion
drivers_df['dob'] = pd.to_datetime(drivers_df['dob'], errors='coerce')
drivers_df['age'] = 2024 - drivers_df['dob'].dt.year

In [ ]:
# Nationality distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

In [ ]:
# Top 10 nationalities
top_nationalities = drivers_df['nationality'].value_counts().head(10)
axes[0].barh(range(len(top_nationalities)), top_nationalities.values)
axes[0].set_yticks(range(len(top_nationalities)))
axes[0].set_yticklabels(top_nationalities.index)
axes[0].set_xlabel('Driver count')
axes[0].set_title('Top 10 driver nationalities')

# Age distribution
axes[1].hist(drivers_df['age'].dropna(), bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Driver count')
axes[1].set_title('Driver age distribution')
axes[1].axvline(drivers_df['age'].mean(), color='red', linestyle='--',
                label=f'Average age: {drivers_df["age"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Circuit analysis
circuits_df = data['circuits']
print("=== CIRCUITS.CSV ===")
print(f"Total circuits: {len(circuits_df)}")
print(f"Countries: {circuits_df['country'].nunique()}")

In [ ]:
# Top 10 countries by number of circuits
top_countries = circuits_df['country'].value_counts().head(10)

plt.figure(figsize=(10, 6))
top_countries.plot(kind='bar')
plt.title('Top 10 countries by number of F1 circuits')
plt.xlabel('Countries')
plt.ylabel('Circuit count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Race analysis
races_df = data['races']
print("=== RACES.CSV ===")
print(f"Total races: {len(races_df)}")
print(f"Years: from {races_df['year'].min()} to {races_df['year'].max()}")

In [ ]:
# Race count by year
races_by_year = races_df['year'].value_counts().sort_index()

plt.figure(figsize=(15, 5))
races_by_year.plot(kind='bar')
plt.title('Race count by year')
plt.xlabel('Year')
plt.ylabel('Race count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Creating the main feature table

def create_base_features():
    """Creating base features for the model"""

    # Starting with results
    df = data['results'].copy()

    print("1. Adding race information...")
    races_info = data['races'][['raceId', 'year', 'round', 'circuitId', 'date']]
    df = df.merge(races_info, on='raceId', how='left')

    print("2. Adding driver information...")
    drivers_info = data['drivers'][['driverId', 'driverRef', 'dob', 'nationality']]
    df = df.merge(drivers_info, on='driverId', how='left')

    print("3. Adding constructor information...")
    constructors_info = data['constructors'][['constructorId', 'constructorRef', 'nationality']]
    df = df.merge(constructors_info, on='constructorId', how='left',
                  suffixes=('_driver', '_constructor'))

    print("4. Adding qualifying positions...")
    quali_info = data['qualifying'][['raceId', 'driverId', 'position']].copy()
    quali_info['quali_position'] = pd.to_numeric(quali_info['position'], errors='coerce')
    df = df.merge(quali_info[['raceId', 'driverId', 'quali_position']],
                  on=['raceId', 'driverId'], how='left')

    print("5. Calculating driver age...")
    df['date'] = pd.to_datetime(df['date'])
    df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
    df['driver_age'] = (df['date'] - df['dob']).dt.days / 365.25

    print("6. Converting finishing position to a number...")
    df['position_num'] = pd.to_numeric(df['position'], errors='coerce')

    print(f"Final dataframe shape: {df.shape}")
    return df

In [ ]:
# Creating the base dataframe
base_df = create_base_features()
print("\nFirst 5 rows of the base dataframe:")
base_df.head()

In [ ]:
# Checking missing values in base features
missing_data = base_df.isnull().sum()
missing_percent = (missing_data / len(base_df)) * 100

missing_df = pd.DataFrame({
    'Missing values': missing_data,
    'Percentage': missing_percent
}).sort_values('Percentage', ascending=False)

print("Missing values in the data:")
print(missing_df[missing_df['Missing values'] > 0])

In [ ]:
# %%
def create_advanced_features(df):
    """Creating advanced features"""

    df = df.copy()

    # Sorting by time
    df = df.sort_values(['year', 'round', 'raceId'])

    print("1. Creating driver rolling average...")
    # Grouping by drivers and calculating cumulative average
    df['driver_avg_history'] = df.groupby('driverId')['position_num']\
        .transform(lambda x: x.expanding().mean().shift(1))

    print("2. Creating constructor rolling average...")
    df['constructor_avg_history'] = df.groupby('constructorId')['position_num']\
        .transform(lambda x: x.expanding().mean().shift(1))

    print("3. Creating rolling average for a specific circuit...")
    # Average on a specific circuit for the driver
    df['circuit_driver_avg'] = df.groupby(['driverId', 'circuitId'])['position_num']\
        .transform(lambda x: x.expanding().mean().shift(1))

    print("4. Creating rolling average for a specific circuit for the constructor...")
    df['circuit_constructor_avg'] = df.groupby(['constructorId', 'circuitId'])['position_num']\
        .transform(lambda x: x.expanding().mean().shift(1))

    print("5. Creating recent form features...")
    # Form over the last 3 races (if enough data is available)
    df['last_3_avg'] = df.groupby('driverId')['position_num']\
        .transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))

    print("6. Creating the difference between qualifying and finish...")
    df['quali_finish_diff'] = df['quali_position'] - df['position_num']

    print("7. Creating experience features...")
    # Count of previous driver races
    df['driver_races_count'] = df.groupby('driverId').cumcount()

    # Count of previous constructor races
    df['constructor_races_count'] = df.groupby('constructorId').cumcount()

    print("8. Creating performance features...")
    # Binary feature: whether the driver was on the podium in the previous race
    df['prev_podium'] = df.groupby('driverId')['position_num']\
        .transform(lambda x: (x.shift(1) <= 3).astype(int))

    # Binary feature: whether there was a win in the previous race
    df['prev_win'] = df.groupby('driverId')['position_num']\
        .transform(lambda x: (x.shift(1) == 1).astype(int))

    return df

# Creating advanced features
print("Creating advanced features...")
advanced_df = create_advanced_features(base_df)

print(f"\nFinal dataframe shape: {advanced_df.shape}")
print(f"Count features: {advanced_df.shape[1]}")

In [ ]:
# Checking created features
feature_columns = ['grid', 'quali_position', 'driver_age', 'driver_avg_history',
                   'constructor_avg_history', 'circuit_driver_avg', 'circuit_constructor_avg',
                   'last_3_avg', 'quali_finish_diff', 'driver_races_count',
                   'constructor_races_count', 'prev_podium', 'prev_win']

print("Statistics for created features:")
advanced_df[feature_columns].describe()

In [ ]:
# Correlation visualization
plt.figure(figsize=(12, 10))

# Selecting numeric features for correlation
numeric_cols = advanced_df.select_dtypes(include=[np.number]).columns
corr_cols = ['position_num', 'grid', 'quali_position', 'driver_age',
             'driver_avg_history', 'constructor_avg_history',
             'circuit_driver_avg', 'driver_races_count']

corr_matrix = advanced_df[corr_cols].corr()

# Correlation heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, fmt='.2f')
plt.title('Feature correlation with the target variable')
plt.tight_layout()
plt.show()

In [ ]:
# Missing value analysis in advanced features
missing_advanced = advanced_df[feature_columns].isnull().sum()
missing_advanced_pct = (missing_advanced / len(advanced_df)) * 100

missing_advanced_df = pd.DataFrame({
    'Feature': missing_advanced.index,
    'Missing values': missing_advanced.values,
    'Percentage': missing_advanced_pct.values
}).sort_values('Percentage', ascending=False)

print("Missing values in advanced features:")
print(missing_advanced_df[missing_advanced_df['Missing values'] > 0])

In [ ]:
# Missing value handling strategy
def handle_missing_values(df):
    """Handling missing values in data"""

    df = df.copy()

    print("Handling missing values:")

    # 1. For history features, fill with the group mean
    history_features = ['driver_avg_history', 'constructor_avg_history',
                       'circuit_driver_avg', 'circuit_constructor_avg', 'last_3_avg']

    for feature in history_features:
        if feature in df.columns:
            # Fill with driver/constructor mean
            if 'driver' in feature:
                df[feature] = df.groupby('driverId')[feature].transform(
                    lambda x: x.fillna(x.mean())
                )
            elif 'constructor' in feature:
                df[feature] = df.groupby('constructorId')[feature].transform(
                    lambda x: x.fillna(x.mean())
                )
            # If missing values remain, fill with the overall mean
            df[feature] = df[feature].fillna(df[feature].mean())
            print(f"  ✓ {feature}: filled")

    # 2. For qualifying position
    if 'quali_position' in df.columns:
        # Fill with starting position
        df['quali_position'] = df['quali_position'].fillna(df['grid'])
        # If starting position is unavailable, fill with the median
        df['quali_position'] = df['quali_position'].fillna(df['quali_position'].median())
        print(f"  ✓ quali_position: filled")

    # 3. For age, fill with the median
    if 'driver_age' in df.columns:
        df['driver_age'] = df['driver_age'].fillna(df['driver_age'].median())
        print(f"  ✓ driver_age: filled")

    # 4. For binary features, fill with 0
    binary_features = ['prev_podium', 'prev_win']
    for feature in binary_features:
        if feature in df.columns:
            df[feature] = df[feature].fillna(0)
            print(f"  ✓ {feature}: filled 0")

    # 5. For the difference between qualifying and finish
    if 'quali_finish_diff' in df.columns:
        df['quali_finish_diff'] = df['quali_finish_diff'].fillna(0)
        print(f"  ✓ quali_finish_diff: filled 0")

    return df

# Applying missing value handling
advanced_df_clean = handle_missing_values(advanced_df)

# Checking that no missing values remain
print(f"\nRemaining missing values: {advanced_df_clean[feature_columns].isnull().sum().sum()}")

In [ ]:
# Outlier analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, feature in enumerate(feature_columns[:6]):
    axes[i].boxplot(advanced_df_clean[feature].dropna())
    axes[i].set_title(feature)
    axes[i].set_ylabel('Value')

plt.tight_layout()
plt.show()

In [ ]:
# Outlier handling (optional for regression)
def remove_outliers(df, columns, n_std=3):
    """Removing outliers using the 3-sigma rule"""

    df_clean = df.copy()
    initial_len = len(df_clean)

    for col in columns:
        if col in df_clean.columns:
            mean = df_clean[col].mean()
            std = df_clean[col].std()

            # Defining bounds
            lower_bound = mean - n_std * std
            upper_bound = mean + n_std * std

            # Removing outliers
            df_clean = df_clean[(df_clean[col] >= lower_bound) &
                                (df_clean[col] <= upper_bound)]

    print(f"Removed {initial_len - len(df_clean)} rows with outliers")
    return df_clean

In [ ]:
# Selecting only finished races for training
final_df = advanced_df_clean[advanced_df_clean['position_num'].notna()].copy()

print(f"Final dataset size: {final_df.shape}")
print(f"Year range: {final_df['year'].min()} - {final_df['year'].max()}")
print(f"Race count: {final_df['raceId'].nunique()}")
print(f"Driver count: {final_df['driverId'].nunique()}")
print(f"Constructor count: {final_df['constructorId'].nunique()}")

In [ ]:
# Saving the final dataset
final_df.to_csv('f1_final_dataset.csv', index=False)
print("Final dataset saved to 'f1_final_dataset.csv'")

# Saving the feature list
features_info = {
    'feature': feature_columns,
    'description': [
        'Starting position',
        'Qualifying position',
        'Driver age',
        'Average driver history',
        'Average constructor history',
        'Driver average on circuit',
        'Constructor average on circuit',
        'Average over the last 3 races',
        'Qualifying-finish difference',
        'Driver experience (number of races)',
        'Constructor experience (number of races)',
        'Podium in the previous race',
        'Win in the previous race'
    ]
}

features_df = pd.DataFrame(features_info)
features_df.to_csv('feature_descriptions.csv', index=False)
print("Feature descriptions saved to 'feature_descriptions.csv'")

In [ ]:
# Splitting data by years for training/validation/testing
years = sorted(final_df['year'].unique())
print(f"Available years: {years}")

# Proposed split:
train_years = [y for y in years if y <= 2017]  # All years up to 2017
val_years = [2018]  # 2018 for validation
test_years = [2019, 2020, 2021, 2022, 2023]  # Most recent years for testing

train_df = final_df[final_df['year'].isin(train_years)]
val_df = final_df[final_df['year'].isin(val_years)]
test_df = final_df[final_df['year'].isin(test_years)]

print(f"\nData split:")
print(f"Train ({len(train_years)} years): {len(train_df)} records")
print(f"Validation ({len(val_years)} years): {len(val_df)} records")
print(f"Test ({len(test_years)} years): {len(test_df)} records")

In [ ]:
# Final dataset statistics
print("=== FINAL DATASET STATISTICS ===")
print(f"Total number of records: {len(final_df)}")
print(f"Target variable: position_num (mean = {final_df['position_num'].mean():.2f})")
print(f"\nTop 5 features by correlation with the target variable:")
correlations = final_df[feature_columns + ['position_num']].corr()['position_num'].sort_values(ascending=False)
for feat in correlations.index[1:6]:
    print(f"  {feat}: {correlations[feat]:.3f}")

In [ ]:
# Final distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Target variable distribution
axes[0,0].hist(final_df['position_num'], bins=20, edgecolor='black', alpha=0.7)
axes[0,0].set_xlabel('Finishing position')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Distribution of finishing positions')
axes[0,0].axvline(final_df['position_num'].mean(), color='red', linestyle='--',
                  label=f'Mean: {final_df["position_num"].mean():.1f}')
axes[0,0].legend()

# 2. Grid vs Position
axes[0,1].scatter(final_df['grid'], final_df['position_num'], alpha=0.3, s=10)
axes[0,1].set_xlabel('Starting position')
axes[0,1].set_ylabel('Finishing position')
axes[0,1].set_title('Starting vs finishing position')

# 3. Distribution by year
year_counts = final_df['year'].value_counts().sort_index()
axes[1,0].bar(year_counts.index, year_counts.values)
axes[1,0].set_xlabel('Year')
axes[1,0].set_ylabel('Count records')
axes[1,0].set_title('Data distribution by year')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Top drivers by number of races
top_drivers = final_df['driverId'].value_counts().head(10)
axes[1,1].barh(range(len(top_drivers)), top_drivers.values)
axes[1,1].set_yticks(range(len(top_drivers)))
axes[1,1].set_yticklabels([f"Driver {d}" for d in top_drivers.index])
axes[1,1].set_xlabel('Race count')
axes[1,1].set_title('Top 10 drivers by number of races')

plt.tight_layout()
plt.show()

In [ ]:
# Saving all important results to a text file for the course project
with open('eda_results.txt', 'w', encoding='utf-8') as f:
    f.write("=== DATA ANALYSIS RESULTS FOR THE COURSE PROJECT ===\n\n")

    f.write("1. GENERAL INFORMATION\n")
    f.write(f"   - Total records in results.csv: {len(results_df)}\n")
    f.write(f"   - Of those, finished races: {results_df['position_num'].notna().sum()}\n")
    f.write(f"   - Finish percentage: {results_df['position_num'].notna().sum()/len(results_df)*100:.1f}%\n")
    f.write(f"   - Count of unique races: {results_df['raceId'].nunique()}\n")
    f.write(f"   - Count of unique drivers: {results_df['driverId'].nunique()}\n")
    f.write(f"   - Count of unique constructors: {results_df['constructorId'].nunique()}\n\n")

    f.write("2. TARGET VARIABLE ANALYSIS\n")
    f.write(f"   - Average finishing position: {results_df['position_num'].mean():.2f}\n")
    f.write(f"   - Median finishing position: {results_df['position_num'].median():.2f}\n")
    f.write(f"   - Standard deviation: {results_df['position_num'].std():.2f}\n")
    f.write(f"   - Minimum: {results_df['position_num'].min()}\n")
    f.write(f"   - Maximum: {results_df['position_num'].max()}\n\n")

    f.write("3. MAIN RETIREMENT REASONS\n")
    for status, count in status_summary.head(5).items():
        f.write(f"   - {status}: {count}\n")
    f.write("\n")

    f.write("4. CORRELATION WITH QUALIFYING\n")
    f.write(f"   - Pearson correlation: {correlation:.3f}\n")
    f.write(f"   - Regression equation: y = {z[0]:.2f}x + {z[1]:.2f}\n\n")

    f.write("5. FINAL DATASET\n")
    f.write(f"   - Shape: {final_df.shape}\n")
    f.write(f"   - Count features: {len(feature_columns)}\n")
    f.write(f"   - Year range: {final_df['year'].min()} - {final_df['year'].max()}\n")
    f.write(f"   - Train set: {len(train_df)} records ({len(train_years)} years)\n")
    f.write(f"   - Validation set: {len(val_df)} records\n")
    f.write(f"   - Test set: {len(test_df)} records\n\n")

    f.write("6. TOP 5 FEATURES BY CORRELATION\n")
    for feat, corr in list(correlations.items())[1:6]:
        f.write(f"   - {feat}: {corr:.3f}\n")

print("Results saved to 'eda_results.txt'")

## Short Conclusion

At this stage, I prepared the working dataset: the data is cleaned, the features are created, and the project is ready for model training.